# Reflect — LLaMA 3 Fine-Tuning

Run this on Colab with **A100 GPU** (Runtime → Change runtime type → A100).

You need:
- HuggingFace token with LLaMA 3 access approved
- HuggingFace repo to push the model to (can be private)

In [ ]:
# Install dependencies
!pip install -q transformers==4.41.0 trl==0.8.6 peft==0.11.1 bitsandbytes==0.43.1 accelerate==0.30.1 datasets==2.19.0

In [ ]:
import os
from google.colab import userdata

# Set these in Colab Secrets (key icon in sidebar)
HF_TOKEN  = userdata.get('HF_TOKEN')
HF_REPO   = userdata.get('HF_REPO')   # e.g. 'shanegraffiti/reflect-llama3'

BASE_MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'
OUT_DIR    = '/content/reflect-llama3-sft'

In [ ]:
# Download and merge all HuggingFace training datasets
import json
import re
from datasets import load_dataset

SYSTEM_PROMPT = """You are Reflect — a trauma-informed analysis tool built on the clinical research of Ramani Durvasula, Jennifer Freyd, Sam Vaknin, Chase Hughes, Joe Navarro, Jessica Taylor, and related scholars in coercive control, betrayal trauma, and psychological abuse.

You exist for one purpose: to help people who are actively being abused, stalked, coercively controlled, or psychologically manipulated understand exactly what is being done to them, why it works, and what it means.

RAMANI DURVASULA: Narcissistic abuse patterns, idealize-devalue-discard cycle, love bombing, narcissistic injury and rage, hoovering, covert vs overt narcissism, entitlement mechanics, why victims stay.

JENNIFER FREYD: Betrayal trauma theory — why victims dissociate from abuse by trusted people. DARVO (Deny Attack Reverse Victim and Offender) as active manipulation. Institutional betrayal. Why naming DARVO disrupts it.

SAM VAKNIN: Narcissistic supply mechanics, primary vs secondary supply, what happens when supply is cut. Narcissistic mortification. The shared fantasy. Why no contact works and why it is attacked.

CHASE HUGHES: Behavioral influence stack, compliance triggers, rapport exploitation, identity anchoring, PEACE model, manufactured vulnerability as a weapon, profile of a manipulator.

JOE NAVARRO: Nonverbal tells — freeze/flight/fight limbic responses, comfort/discomfort signals, territorial behavior, deception reads.

JESSICA TAYLOR: Victim-blaming as systemic mechanism, how mental health systems retraumatize survivors, trauma responses as rational adaptations, misuse of diagnosis to silence victims.

OPERATE: Require concrete behavioral specifics. When a pattern is textbook, say so directly. Name tactics and explain the mechanism. Never hedge clear findings. Never use therapist language. Clinical, direct, peer-level."""

def fmt(instruction, output):
    if not instruction or not output:
        return None
    instruction, output = str(instruction).strip(), str(output).strip()
    if len(instruction) < 20 or len(output) < 30:
        return None
    return {
        'text': (
            f'<|begin_of_text|>'
            f'<|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>'
            f'<|start_header_id|>user<|end_header_id|>\n\n{instruction}<|eot_id|>'
            f'<|start_header_id|>assistant<|end_header_id|>\n\n{output}<|eot_id|>'
        )
    }

records = []

sources = [
    ('Amod/mental_health_counseling_conversations', 'train', lambda r: fmt(r.get('Context'), r.get('Response'))),
    ('fadodr/mental_health_therapy', 'train', lambda r: fmt(r.get('instruction'), r.get('output'))),
    ('ShenLab/MentalChat16K', 'train', lambda r: fmt(r.get('instruction') or r.get('input') or r.get('question'), r.get('output') or r.get('response'))),
    ('nbertagnolli/counsel-chat', 'train', lambda r: fmt((r.get('questionTitle','') + ' ' + r.get('questionText','')).strip(), r.get('answerText'))),
    ('mpingale/mental-health-chat-dataset', 'train', lambda r: fmt((r.get('questionTitle','') + ' ' + r.get('questionText','')).strip(), r.get('answerText'))),
]

for repo, split, fn in sources:
    print(f'Loading {repo}...')
    try:
        ds = load_dataset(repo, split=split)
        batch = [r for row in ds if (r := fn(row))]
        records.extend(batch)
        print(f'  {len(batch):,} records')
    except Exception as e:
        print(f'  SKIP: {e}')

# Large dataset — sample 50K to keep training time reasonable on first run
print('Loading IINOVAII/therapy-conversations-combined (sampling 50K)...')
try:
    ds = load_dataset('IINOVAII/therapy-conversations-combined', split='train')
    batch = []
    for row in ds:
        inst = ((row.get('instruction') or '') + ' ' + (row.get('input') or '')).strip()
        r = fmt(inst, row.get('output'))
        if r:
            batch.append(r)
        if len(batch) >= 50000:
            break
    records.extend(batch)
    print(f'  {len(batch):,} records')
except Exception as e:
    print(f'  SKIP: {e}')

import random
random.seed(42)
random.shuffle(records)

# Deduplicate
import hashlib
seen = set()
unique = []
for r in records:
    h = hashlib.sha256(r['text'][:200].encode()).hexdigest()
    if h not in seen:
        seen.add(h)
        unique.append(r)

print(f'\nTotal unique records: {len(unique):,}')

split_idx = int(len(unique) * 0.95)
train_records = unique[:split_idx]
eval_records  = unique[split_idx:]
print(f'Train: {len(train_records):,}  Eval: {len(eval_records):,}')

In [ ]:
# Write to disk for the Datasets loader
with open('/content/train.jsonl', 'w') as f:
    for r in train_records:
        f.write(json.dumps(r) + '\n')
with open('/content/eval.jsonl', 'w') as f:
    for r in eval_records:
        f.write(json.dumps(r) + '\n')
print('Wrote train.jsonl and eval.jsonl')

In [ ]:
import torch
from datasets import load_dataset as lds
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {BASE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
train_ds = lds('json', data_files='/content/train.jsonl', split='train')
eval_ds  = lds('json', data_files='/content/eval.jsonl',  split='train')

training_args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim='paged_adamw_32bit',
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    bf16=True,
    max_grad_norm=0.3,
    evaluation_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=50,
    report_to='none',
    group_by_length=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field='text',
    max_seq_length=2048,
    packing=True,
)

print('Starting training...')
trainer.train()

In [ ]:
# Save and push to HuggingFace Hub
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

from peft import PeftModel
from transformers import AutoModelForCausalLM

print('Merging LoRA weights...')
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, token=HF_TOKEN, torch_dtype=torch.bfloat16, device_map='cpu')
merged = PeftModel.from_pretrained(base, OUT_DIR)
merged = merged.merge_and_unload()

print(f'Pushing to {HF_REPO}...')
merged.push_to_hub(HF_REPO, token=HF_TOKEN, private=True)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN, private=True)
print(f'Done. Model at: https://huggingface.co/{HF_REPO}')